In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error



In [2]:
# Load and clean dataset
df = pd.read_csv("uber.csv")
df = df.dropna()
df = df[(df["fare_amount"] > 0) & (df["passenger_count"].between(1, 6))]



In [3]:
for col in ["pickup_latitude", "dropoff_latitude"]:
    df = df[df[col].between(-90, 90)]

for col in ["pickup_longitude", "dropoff_longitude"]:
    df = df[df[col].between(-180, 180)]



In [4]:
coordinate_cols = ["pickup_latitude", "pickup_longitude",
                   "dropoff_latitude", "dropoff_longitude"]

df = df[(df[coordinate_cols] != 0).all(axis=1)].copy()

# Calculate Haversine distance
lat1 = np.radians(df["pickup_latitude"])
lat2 = np.radians(df["dropoff_latitude"])
dlat = lat2 - lat1
dlon = np.radians(df["dropoff_longitude"] - df["pickup_longitude"])

a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
df["distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))

df = df[df["distance_km"].between(0.1, 100)].copy()
print("Usable rides:", len(df))



Usable rides: 192027


In [5]:
# Select features and target
features = ["distance_km", "passenger_count", "pickup_latitude",
            "pickup_longitude", "dropoff_latitude", "dropoff_longitude"]

X = df[features]
y = df["fare_amount"]



In [6]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [7]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [8]:
# Model without PCA
model_original = LinearRegression()
model_original.fit(X_train_scaled, y_train)
pred_original = model_original.predict(X_test_scaled)



In [9]:
# Apply PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)




In [10]:
# Model with PCA
model_pca = LinearRegression()
model_pca.fit(X_train_pca, y_train)
pred_pca = model_pca.predict(X_test_pca)



In [11]:
# Compare models
print("WITHOUT PCA")
print("Features:", X_train_scaled.shape[1])
print("R2:", round(r2_score(y_test, pred_original), 3))
print("MAE:", round(mean_absolute_error(y_test, pred_original), 3))
print("RMSE:", round(mean_squared_error(y_test, pred_original)**0.5, 3))

print("\nWITH PCA")
print("Components:", X_train_pca.shape[1])
print("Explained variance (%):", round(pca.explained_variance_ratio_.sum() * 100, 2))
print("R2:", round(r2_score(y_test, pred_pca), 3))
print("MAE:", round(mean_absolute_error(y_test, pred_pca), 3))
print("RMSE:", round(mean_squared_error(y_test, pred_pca)**0.5, 3))

WITHOUT PCA
Features: 6
R2: 0.774
MAE: 2.305
RMSE: 4.485

WITH PCA
Components: 3
Explained variance (%): 99.77
R2: 0.772
MAE: 2.313
RMSE: 4.505
